In [1]:
!pip install datasets

In [2]:
#load dataset
from datasets import load_dataset

dataset = load_dataset("Henil1/Sans-eng")
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['S.No.', 'Title', 'Chapter', 'Verse', 'Sanskrit Anuvad', 'Hindi Anuvad', 'Enlgish Translation'],
        num_rows: 700
    })
})
{'S.No.': 1, 'Title': "Arjuna's Vishada Yoga", 'Chapter': 'Chapter 1', 'Verse': 'Verse 1.1', 'Sanskrit Anuvad': 'धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...', 'Hindi Anuvad': 'धृतराष्ट्र बोले- हे संजय! धर्मभूमि कुरुक्षेत्र में एकत्रित, युद्ध की इच्छावाले मेरे और पाण्डु के पुत्रों ने क्या किया?', 'Enlgish Translation': 'Dhrtarashtra asked of Sanjaya: O SANJAYA, what did my warrior sons and those of Pandu do when they were gathered at KURUKSHETRA, the field of religious activities? Tell me of those happenings.'}


In [3]:
print(dataset['train'])

Dataset({
    features: ['S.No.', 'Title', 'Chapter', 'Verse', 'Sanskrit Anuvad', 'Hindi Anuvad', 'Enlgish Translation'],
    num_rows: 700
})


In [4]:
#create pandas dataframe for data manipulation
df = dataset['train'].to_pandas()

In [5]:
#create new merged column for storing
df['merged'] = df['Sanskrit Anuvad'] + '|' + df['Enlgish Translation']

In [6]:
print(df['merged'])

0      धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे सम...
1      सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं व्यूढं दु...
2      पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् । व...
3      अत्र शूरा महेष्वासा भीमार्जुनसमा युधि । युयुधा...
4      धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् । पुर...
                             ...                        
695    संजय उवाच । इत्यहं वासुदेवस्य पार्थस्य च महात्...
696    व्यासप्रसादाच्छ्रुतवानेतद्गुह्यमहं परम् । योगं...
697    राजन्संस्मृत्य संस्मृत्य संवादमिममद्भुतम् । के...
698    तच्च संस्मृत्य संस्मृत्य रूपमत्यद्भुतं हरेः । ...
699    यत्र योगेश्वरः कृष्णो यत्र पार्थो धनुर्धरः । त...
Name: merged, Length: 700, dtype: object


In [7]:
print(df['merged'][0])

धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...|Dhrtarashtra asked of Sanjaya: O SANJAYA, what did my warrior sons and those of Pandu do when they were gathered at KURUKSHETRA, the field of religious activities? Tell me of those happenings.


In [8]:
#setup groq
import os

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [9]:
!pip install langchain_groq

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = 'llama-3.1-8b-instant', temperature = 0.2)
prompt = """You will be give Bhagwad Gita verse. Your task is to generate a simple question in English,
that a person would ask to find the verse. The main target audience is Indian people, so try generating
questions in that language, not hard rule, in that direction. Also, the question should be quite simple
and not feel AI generated with loaded words. You need to return only the question, and nothing else."""
pairs = []

for idx, item in df.iterrows():
  answer = item['Enlgish Translation']
  response = llm.invoke(prompt + "\n\n" + answer)
  pairs.append({"query": response.content, "document": item['merged']})


KeyboardInterrupt: 

In [12]:
print(len(pairs))

16


In [13]:
import json
with open('pairs.json', 'r', encoding='utf-8') as f:
    pairs = json.load(f)
print(len(pairs))

700


In [14]:
#download JSON
from google.colab import files
files.download('pairs.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import json
with open('pairs.json', 'w', encoding='utf-8') as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)
print(f"Saved {len(pairs)} pairs")

Saved 700 pairs


In [16]:
print(len(pairs))
print(pairs[0])
print(pairs[699])

700
{'query': 'Kya Pandavon aur mera putra Kurukshetra mein kya kiya?', 'document': 'धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...|Dhrtarashtra asked of Sanjaya: O SANJAYA, what did my warrior sons and those of Pandu do when they were gathered at KURUKSHETRA, the field of religious activities? Tell me of those happenings.'}
{'query': 'Krishna aur Arjun ke beech kya baat hai jo sabse achha hai?', 'document': 'यत्र योगेश्वरः कृष्णो यत्र पार्थो धनुर्धरः । तत्र श्रीर्विजयो भूतिर्ध्रुवा नीतिर्मतिर्मम ॥ १८.७८ ॥|Wherever there is the Divine Lord Krishna, the Master of all Yoga, and the able disciple Arjuna, there is beauty, morality, extraordinary power, and victory over all evil. O King Dhrtarastra, this is my unshakeable belief and faith.'}


In [17]:
#Embeddings

In [18]:
# Install
!pip install chromadb sentence-transformers

# Imports
import chromadb
from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Get all documents from pairs
documents = [pair['document'] for pair in pairs]

# Embed all documents
doc_embeddings = model.encode(documents, show_progress_bar=True)

# Store in ChromaDB
client = chromadb.Client()
collection = client.create_collection("gita_baseline")

collection.add(
    documents=documents,
    embeddings=doc_embeddings.tolist(),
    ids=[str(i) for i in range(len(documents))]
)

print(f"Stored {collection.count()} documents in ChromaDB")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Stored 700 documents in ChromaDB


In [19]:
#Baseline Evaluation

In [20]:
queries = [pair['query'] for pair in pairs]
correct = 0

for i, pair in enumerate(pairs):
  query = pair['query']
  correct_doc = pair['document']

  # Embed the individual query using same model
  query_embedding = model.encode([query]).tolist()

  # Search ChromaDB - get top 5
  results = collection.query(
        query_embeddings=query_embedding,
        n_results=5
  )

  # Check if correct document is in top 5
  retrieved_docs = results['documents'][0]
  if correct_doc in retrieved_docs:
      correct += 1

recall_at_5 = correct / len(pairs)
print(f"Baseline Recall@5: {recall_at_5:.2%}")
print(f"Correct: {correct} / {len(pairs)}")

Baseline Recall@5: 2.57%
Correct: 18 / 700


In [21]:
#Fine Tuning - 1

In [23]:
from sentence_transformers import SentenceTransformer, InputExample

train_examples = []
for pair in pairs:
    train_examples.append(InputExample(texts=[pair['query'], pair['document']]))

In [24]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [25]:
#loss function
from sentence_transformers import losses
train_loss = losses.MultipleNegativesRankingLoss(model)

/tmp/ipykernel_17638/2756113841.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


In [27]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=10
)

Step,Training Loss


In [28]:
model.save('sanskrit-retrieval-finetuned')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
#save in local
from google.colab import files
import shutil

shutil.make_archive('sanskrit-retrieval-finetuned', 'zip', 'sanskrit-retrieval-finetuned')
files.download('sanskrit-retrieval-finetuned.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# Re-embed with fine-tuned model
doc_embeddings_finetuned = model.encode(documents, show_progress_bar=True)

# New ChromaDB collection
collection_finetuned = client.create_collection("gita_finetuned")
collection_finetuned.add(
    documents=documents,
    embeddings=doc_embeddings_finetuned.tolist(),
    ids=[str(i) for i in range(len(documents))]
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [31]:
queries = [pair['query'] for pair in pairs]
correct = 0

for i, pair in enumerate(pairs):
  query = pair['query']
  correct_doc = pair['document']

  # Embed the individual query using same model
  query_embedding = model.encode([query]).tolist()

  # Search ChromaDB - get top 5
  results = collection_finetuned.query(
        query_embeddings=query_embedding,
        n_results=5
  )

  # Check if correct document is in top 5
  retrieved_docs = results['documents'][0]
  if correct_doc in retrieved_docs:
      correct += 1

recall_at_5 = correct / len(pairs)
print(f"Fine-tuned Recall@5: {recall_at_5:.2%}")
print(f"Correct: {correct} / {len(pairs)}")

Fine-tuned Recall@5: 26.86%
Correct: 188 / 700


In [32]:
#Fine Tuning - 2

In [33]:
model_v2 = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

model_v2.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=15,
    warmup_steps=10
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
500,0.974367


In [34]:
model_v2.save('sanskrit-retrieval-finetuned2')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
#save in local
from google.colab import files
import shutil

shutil.make_archive('sanskrit-retrieval-finetuned2', 'zip', 'sanskrit-retrieval-finetuned2')
files.download('sanskrit-retrieval-finetuned2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
# Re-embed with fine-tuned model2
doc_embeddings_finetuned = model_v2.encode(documents, show_progress_bar=True)

# New ChromaDB collection
collection_finetuned = client.create_collection("gita_finetuned2")
collection_finetuned.add(
    documents=documents,
    embeddings=doc_embeddings_finetuned.tolist(),
    ids=[str(i) for i in range(len(documents))]
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [37]:
queries = [pair['query'] for pair in pairs]
correct = 0

for i, pair in enumerate(pairs):
  query = pair['query']
  correct_doc = pair['document']

  # Embed the individual query using same model
  query_embedding = model_v2.encode([query]).tolist()

  # Search ChromaDB - get top 5
  results = collection_finetuned.query(
        query_embeddings=query_embedding,
        n_results=5
  )

  # Check if correct document is in top 5
  retrieved_docs = results['documents'][0]
  if correct_doc in retrieved_docs:
      correct += 1

recall_at_5 = correct / len(pairs)
print(f"Fine-tuned Recall@5: {recall_at_5:.2%}")
print(f"Correct: {correct} / {len(pairs)}")

Fine-tuned Recall@5: 80.71%
Correct: 565 / 700


In [39]:
def rag(query, top_k = 5):

  #embed the query
  query_embedding_final = model_v2.encode([query]).tolist()

  #retrieve top_k chunks
  results = collection_finetuned.query(
      query_embeddings=query_embedding_final,
      n_results=top_k
  )

  #extract documents
  retrieved_docs_final = results['documents'][0]
  context ="\n\n".join(retrieved_docs_final)

  # Step 4: Build prompt and call Groq
  prompt = f"""You are a Bhagavad Gita expert. Answer the user's question based only on the provided verses.

Verses:
{context}

Question: {query}

Answer:"""

  response = llm.invoke(prompt)
  return response.content

# Test
print(rag("What does Gita say about karma?"))

According to the provided verses from the Bhagavad Gita, the following points are made about karma:

1. **Understanding Karma**: The Gita emphasizes the importance of understanding the principle of karma, which is complex and mysterious (Verse 4.17).

2. **Right to Perform Actions**: Arjuna is told that he has the right to perform his actions, duties, and responsibilities in life, but not to desire the results of those actions (Verse 2.47). This is the law of karma that one should always obey.

3. **Karmyoga**: The Gita describes karmyoga as a path where one performs actions without attachment to their results, neither desiring nor fearing them (Verse 2.47).

4. **Divine Karma**: Lord Krishna states that his own actions are divine and that those who understand this truth are not reborn after leaving their bodies (Verse 4.9).

5. **Consequences of Karma**: The Gita explains that those who perform attached karma (actions performed for personal gain) are reborn into the cycle of birth and

In [41]:
print(rag("What does Gita say about karma?"))

According to the provided verses, the Bhagavad Gita says the following about karma:

1. One should know the principle of karma (action), akarma (inaction), and forbidden karma (forbidden action). The philosophy behind karma is very deep and mysterious. (Verse 4.17)

2. One has the right to perform their actions, duties, and responsibilities in life, but the results of these actions should not concern them. (Verse 2.47)

3. The results of actions are not in one's hands but in the hands of the Lord. Therefore, one should not desire results for their actions. (Verse 2.47)

4. The Gita emphasizes the importance of performing actions without attachment to their results. This is known as Karmyoga. (Verse 2.47)

5. The Gita also mentions that one's karma can lead to rebirth in the world (subject to the cycle of birth and death) once their good karma has expired. (Verse 4.9 and the additional verse)

6. The Gita states that the one who knows and understands the divine nature of their actions (

In [42]:
#Failure analysis

In [43]:
failures = []
for i, pair in enumerate(pairs):
    query = pair['query']
    correct_doc = pair['document']
    query_embedding = model_v2.encode([query]).tolist()
    results = collection_finetuned.query(query_embeddings=query_embedding, n_results=5)
    retrieved_docs = results['documents'][0]
    if correct_doc not in retrieved_docs:
        failures.append({"query": query, "correct": correct_doc, "retrieved": retrieved_docs[0]})

print(f"Total failures: {len(failures)}")
print("\nExample failures:")
for f in failures[:3]:
    print(f"\nQuery: {f['query']}")
    print(f"Expected: {f['correct'][:100]}...")
    print(f"Got: {f['retrieved'][:100]}...")

Total failures: 135

Example failures:

Query: Kya Arjun ko yuddh ke samay kaisa darshana hua?
Expected: ततः शङ्खाश्च भेर्यश्च पणवानकगोमुखाः । सहसैवाभ्यहन्यन्त स शब्दस्तुमुलोऽभवत् ॥ १.१३ ॥|Tremendous noise...
Got: अत्र शूरा महेष्वासा भीमार्जुनसमा युधि । युयुधानो विराटश्च द्रुपदश्च महारथः ॥ १.४ ॥|Present here are ...

Query: Krishna aur Arjun ke beech kya baat hui thi?
Expected: अथ व्यवस्थितान्दृष्ट्वा धार्तराष्ट्रान्कपिध्वजः प्रवृत्ते शस्त्रसम्पाते धनुरुद्यम्य पाण्डवः हृषीके.....
Got: यत्र योगेश्वरः कृष्णो यत्र पार्थो धनुर्धरः । तत्र श्रीर्विजयो भूतिर्ध्रुवा नीतिर्मतिर्मम ॥ १८.७८ ॥|W...

Query: Arjun ne kya kaha tha?
Expected: हृषीकेशं तदा वाक्यमिदमाह महीपते |अर्जुन उवाच |सेनयोरुभयोर्मध्ये रथं स्थापय मेऽच्युत ॥ १.२१ ॥|Arjuna ...
Got: इति ते ज्ञानमाख्यातं गुह्याद्\u200cगुह्यतरं मया । विमृश्यैतदशेषेण यथेच्छसि तथा कुरु ॥ १८.६३ ॥|O Arju...
